# S11 - BI/ML distribuido sobre el tráfico de red (Proyecto Sello)

Actividad autónoma de S11 (4.1). Dos patrones de publicación hacia el mismo Pushgateway (`obs/`, `:9091`):

- **Parte A (BI batch, una vez):** KPIs de negocio sobre evidencia ya persistida: el Parquet de observabilidad de S9
  (`red-telemetria`, el flujo IoT de mi PC) y las predicciones en streaming de S10 (tráfico del campus).
- **Parte B (ML en vivo, continuo):** el `PipelineModel` de S10 recargado **sin reentrenar**, aplicado al topic
  `trafico-ventanas` mientras `uso-trafico/ventanas-red-py` publica. En cada micro-lote se empujan el valor real, la
  predicción t+1 y el error de la predicción anterior (la comparación "en vivo" de la Regla #37).

| Tarea 4.1 | Sección |
|---|---|
| 1. Dos o más KPIs sobre evidencia propia, publicados una vez | A.1 - A.3 |
| 2. Modelo propio redesplegado en vivo, publicando por micro-lote | B.1 - B.3 |
| 3. Dos o más consultas PromQL separadas en el tiempo con valor distinto | B.4 |
| 4. Tablero de Grafana | `obs/grafana/dashboards/s11-bi-ml.json` |

In [1]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window

spark = (
    SparkSession.builder
    .appName("s11-bi-ml-trafico")
    .master("local[*]")
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.13:4.2.0")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.session.timeZone", "America/Lima")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

:: loading settings :: url = jar:file:/usr/local/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
org.apache.spark#spark-sql-kafka-0-10_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-66b811cc-b4d8-44b7-9242-6dda07d46881;1.0
	confs: [default]


	found org.apache.spark#spark-sql-kafka-0-10_2.13;4.2.0 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.13;4.2.0 in central
	found org.apache.kafka#kafka-clients;3.9.2 in central
	found at.yawk.lz4#lz4-java;1.11.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.8 in central
	found org.slf4j#slf4j-api;2.0.17 in central
	found org.apache.hadoop#hadoop-client-runtime;3.5.0 in central
	found org.apache.hadoop#hadoop-client-api;3.5.0 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.scala-lang.modules#scala-parallel-collections_2.13;1.2.0 in central


	found org.apache.commons#commons-pool2;2.13.1 in central
:: resolution report :: resolve 344ms :: artifacts dl 9ms
	:: modules in use:
	at.yawk.lz4#lz4-java;1.11.0 from central in [default]
	com.google.code.findbugs#jsr305;3.0.0 from central in [default]
	org.apache.commons#commons-pool2;2.13.1 from central in [default]
	org.apache.hadoop#hadoop-client-api;3.5.0 from central in [default]
	org.apache.hadoop#hadoop-client-runtime;3.5.0 from central in [default]
	org.apache.kafka#kafka-clients;3.9.2 from central in [default]
	org.apache.spark#spark-sql-kafka-0-10_2.13;4.2.0 from central in [default]
	org.apache.spark#spark-token-provider-kafka-0-10_2.13;4.2.0 from central in [default]
	org.scala-lang.modules#scala-parallel-collections_2.13;1.2.0 from central in [default]
	org.slf4j#slf4j-api;2.0.17 from central in [default]
	org.xerial.snappy#snappy-java;1.1.10.8 from central in [default]
	---------------------------------------------------------------------
	|                  |        

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


Spark 4.2.0


In [2]:
import json, os, shutil, time, urllib.parse, urllib.request
from datetime import datetime, timedelta, timezone

PUSHGATEWAY = "http://pushgateway:9091"
PROMETHEUS = "http://prometheus:9090"
LIMA = timezone(timedelta(hours=-5))

def hora(ms=None):
    return datetime.fromtimestamp((ms if ms is not None else time.time() * 1000) / 1000, LIMA).strftime("%H:%M:%S")

def promql(consulta):
    url = f"{PROMETHEUS}/api/v1/query?" + urllib.parse.urlencode({"query": consulta})
    r = json.load(urllib.request.urlopen(url, timeout=10))["data"]["result"]
    return float(r[0]["value"][1]) if r else None

def push(grupo, metricas, metodo="PUT"):
    cuerpo = "".join(f"# TYPE {n} gauge\n{n} {v}\n" for n, v in metricas.items())
    req = urllib.request.Request(f"{PUSHGATEWAY}/metrics/{grupo}", data=cuerpo.encode(), method=metodo)
    with urllib.request.urlopen(req, timeout=10) as r:
        return r.status

## Parte A — BI batch

### A.1 KPIs del flujo `red-telemetria` (Parquet de observabilidad de S9)

In [3]:
RUTA_S9 = "../s09-observabilidad-red/artifacts/red_observabilidad_parquet"
obs9 = spark.read.parquet(RUTA_S9)

kpi9 = obs9.agg(
    F.count("*").alias("eventos"),
    F.avg(F.col("isValid").cast("int")).alias("validosRatio"),
    F.avg(F.col("fueraDeRango").cast("int")).alias("fueraDeRangoRatio"),
    F.avg("latencyMs").alias("latenciaPromedioMs"),
    F.expr("percentile_approx(latencyMs, 0.95)").alias("latenciaP95Ms"),
    F.countDistinct("sensorId").alias("sensores"),
    F.date_format(F.min("kafkaTimestamp"), "dd/MM/yyyy HH:mm:ss").alias("desde"),
    F.date_format(F.max("kafkaTimestamp"), "dd/MM/yyyy HH:mm:ss").alias("hasta"),
).first()

eth = obs9.filter(F.col("sensorId") == "nsmj-ethernet")
kpi_eth = eth.agg(
    F.avg("bytesRxPorSeg").alias("bytesRxPromedio"),
    F.max(F.greatest("bytesRxPorSeg", "bytesTxPorSeg") / (F.col("velocidadEnlaceMbps") * 125000) * 100).alias("usoMaxPct"),
).first()

print(f"Evidencia S9: {kpi9['eventos']:,} lecturas de {kpi9['sensores']} sensores, {kpi9['desde']} -> {kpi9['hasta']}")
print(f"  Eventos válidos .............. {kpi9['validosRatio']:.4f}")
print(f"  Fuera de rango físico ........ {kpi9['fueraDeRangoRatio']:.4f}")
print(f"  Latencia promedio / p95 ...... {kpi9['latenciaPromedioMs']:.1f} ms / {kpi9['latenciaP95Ms']} ms")
print(f"  nsmj-ethernet: tráfico medio {kpi_eth['bytesRxPromedio']:,.0f} B/s, uso máximo del enlace {kpi_eth['usoMaxPct']:.2f} %")

Evidencia S9: 1,704 lecturas de 4 sensores, 15/10/2026 20:27:17 -> 15/10/2026 20:34:34
  Eventos válidos .............. 1.0000
  Fuera de rango físico ........ 0.0000
  Latencia promedio / p95 ...... 2489.3 ms / 4753 ms
  nsmj-ethernet: tráfico medio 1,044,716 B/s, uso máximo del enlace 56.43 %


### A.2 KPIs del tráfico del campus y del modelo (predicciones persistidas de S10)

In [4]:
RUTA_S10_PRED = "../s10-series-tiempo-red/artifacts/predicciones_stream"
pred10 = spark.read.parquet(RUTA_S10_PRED)
auditado = (pred10.withColumn("realSiguienteBps", F.lead("bytesPorSeg").over(Window.orderBy("ventana")))
                  .na.drop(subset=["realSiguienteBps"]))
kpi10 = pred10.agg(F.count("*").alias("ventanas"), F.avg("bytesPorSeg").alias("bpsPromedio"),
                   F.max("bytesPorSeg").alias("bpsPico"),
                   F.date_format(F.min("ventana"), "dd/MM/yyyy HH:mm").alias("desde"),
                   F.date_format(F.max("ventana"), "dd/MM/yyyy HH:mm").alias("hasta")).first()
mae10 = auditado.select(F.avg(F.abs(F.col("prediccionSiguienteBps") - F.col("realSiguienteBps")))).first()[0]

print(f"Evidencia S10: {kpi10['ventanas']} ventanas de 5 min del campus, {kpi10['desde']} -> {kpi10['hasta']}")
print(f"  Tráfico promedio / pico ...... {kpi10['bpsPromedio']:.1f} / {kpi10['bpsPico']:.1f} B/s")
print(f"  MAE del modelo en producción . {mae10:.1f} B/s  ({auditado.count()} predicciones auditadas contra el real siguiente)")

Evidencia S10: 120 ventanas de 5 min del campus, 14/08/2026 08:00 -> 14/08/2026 17:55
  Tráfico promedio / pico ...... 65.7 / 269.0 B/s


  MAE del modelo en producción . 24.0 B/s  (119 predicciones auditadas contra el real siguiente)


### A.3 Publicar los KPIs una sola vez (`PUT`) y verificarlos en Prometheus

In [5]:
kpis = {
    "lambda26_bi_red_eventos_total": kpi9["eventos"],
    "lambda26_bi_red_validos_ratio": round(kpi9["validosRatio"], 6),
    "lambda26_bi_red_fuera_rango_ratio": round(kpi9["fueraDeRangoRatio"], 6),
    "lambda26_bi_red_latencia_promedio_ms": round(kpi9["latenciaPromedioMs"], 2),
    "lambda26_bi_red_latencia_p95_ms": kpi9["latenciaP95Ms"],
    "lambda26_bi_red_bytes_seg_promedio": round(kpi_eth["bytesRxPromedio"], 2),
    "lambda26_bi_red_uso_enlace_max_pct": round(kpi_eth["usoMaxPct"], 4),
    "lambda26_bi_campus_bytes_seg_promedio": round(kpi10["bpsPromedio"], 3),
    "lambda26_bi_campus_bytes_seg_pico": round(kpi10["bpsPico"], 3),
    "lambda26_bi_campus_mae_produccion_bps": round(mae10, 3),
}
GRUPO_BATCH = "job/lambda26_bi_flujo/instance/s11_batch"
print(f"{hora()}  PUT {GRUPO_BATCH} -> HTTP {push(GRUPO_BATCH, kpis)}  ({len(kpis)} métricas)")

inmediato = {n: promql(n) for n in kpis}
print(f"{hora()}  consulta inmediata a Prometheus: {sum(v is None for v in inmediato.values())} de {len(kpis)} métricas SIN DATOS")
t0 = time.time()
while any(promql(n) is None for n in kpis) and time.time() - t0 < 60:
    time.sleep(2)
print(f"{hora()}  todas visibles en Prometheus tras {time.time() - t0:.0f} s (próximo scrape del Pushgateway):")
for n in kpis:
    print(f"   {n:42s} {promql(n)}")

19:33:53  PUT job/lambda26_bi_flujo/instance/s11_batch -> HTTP 200  (10 métricas)
19:33:53  consulta inmediata a Prometheus: 0 de 10 métricas SIN DATOS
19:33:53  todas visibles en Prometheus tras 0 s (próximo scrape del Pushgateway):
   lambda26_bi_red_eventos_total              1704.0
   lambda26_bi_red_validos_ratio              1.0
   lambda26_bi_red_fuera_rango_ratio          0.0
   lambda26_bi_red_latencia_promedio_ms       2489.29
   lambda26_bi_red_latencia_p95_ms            4753.0
   lambda26_bi_red_bytes_seg_promedio         1044716.38
   lambda26_bi_red_uso_enlace_max_pct         56.4261
   lambda26_bi_campus_bytes_seg_promedio      65.683
   lambda26_bi_campus_bytes_seg_pico          269.017
   lambda26_bi_campus_mae_produccion_bps      23.988


## Parte B — ML en vivo

### B.1 Recargar el modelo de S10, sin reentrenar

In [6]:
from pyspark.ml import PipelineModel

RUTA_MODELO_S10 = "../s10-series-tiempo-red/artifacts/models/modelo_trafico_5min"
modelo = PipelineModel.load(RUTA_MODELO_S10)
VARIABLES = modelo.stages[0].getInputCols()
print("Modelo cargado:", [type(s).__name__ for s in modelo.stages])
print(f"Espera {len(VARIABLES)} variables:", VARIABLES)
print("Coeficientes (los mismos de S10):", [round(c, 4) for c in modelo.stages[1].coefficients.toArray()[:5]], "...")

Modelo cargado: ['VectorAssembler', 'LinearRegressionModel']
Espera 17 variables: ['l1', 'l2', 'l3', 'l6', 'l12', 'm6', 'm12', 'sd6', 'f1', 'f2', 'f3', 'mf6', 'tcpRatio', 'webRatio', 'hSin', 'hCos', 'finde']
Coeficientes (los mismos de S10): [np.float64(0.1663), np.float64(0.0838), np.float64(0.0058), np.float64(-0.0768), np.float64(-0.0037)] ...


### B.2 Leer `trafico-ventanas` y aplicar el modelo, sin ventanas ni estado

In [7]:
from pyspark.sql.types import StructType, StringType, DoubleType, LongType

esquema = (StructType().add("tipoEvento", StringType()).add("sensorId", StringType())
           .add("ventanaInicio", LongType()).add("bytesPorSeg", DoubleType()).add("flujos", LongType()))
for v in VARIABLES:
    esquema = esquema.add(v, DoubleType())
esquema = esquema.add("origen", StringType()).add("timestamp", LongType())

eventos = (spark.readStream.format("kafka").option("kafka.bootstrap.servers", "kafka:9092")
           .option("subscribe", "trafico-ventanas").option("startingOffsets", "latest")
           .option("failOnDataLoss", "false").load()
           .select(F.from_json(F.col("value").cast("string"), esquema).alias("e")).select("e.*")
           .withColumn("ventana", F.timestamp_millis("ventanaInicio")))
predicciones = modelo.transform(eventos)
print("es un stream:", predicciones.isStreaming)

es un stream: True


### B.3 Publicar en cada micro-lote: real, predicción t+1 y error de la predicción anterior

El error se calcula en el *driver*, comparando la predicción que se publicó para esta ventana (en el micro-lote
anterior) con lo que de verdad llegó ahora. El modelo sigue sin estado: lo único que se recuerda es la última predicción,
para monitorearla.

In [8]:
RUTA_PRED = "./artifacts/predicciones_en_vivo"
CHK = "./artifacts/chk_predicciones_en_vivo"
for ruta in (RUTA_PRED, CHK, "./artifacts/stream_listo"):
    shutil.rmtree(ruta, ignore_errors=True) if os.path.isdir(ruta) else (os.path.exists(ruta) and os.remove(ruta))
os.makedirs("./artifacts", exist_ok=True)

GRUPO_VIVO = "job/lambda26_ml_streaming/instance/campus-trcu"
monitor = {"prediccion_anterior": None, "pushes": []}

def microlote(df, batch_id):
    # Las horas se formatean en Spark (zona America/Lima): collect() las devolvería en la zona del contenedor (UTC)
    sel = (df.select("ventana", F.date_format("ventana", "dd/MM HH:mm").alias("ventanaTxt"),
                     F.unix_timestamp("ventana").alias("ventanaSeg"), "bytesPorSeg",
                     F.expm1("prediction").alias("prediccionSiguienteBps"))
             .orderBy("ventana"))
    filas = sel.collect()
    if not filas:
        monitor["pushes"].append((batch_id, hora(), None, None, None, "sin filas"))
        return
    sel.drop("ventanaTxt").withColumn("batchId", F.lit(batch_id)).write.mode("append").parquet(RUTA_PRED)
    for f in filas:
        anterior = monitor["prediccion_anterior"]
        error = abs(anterior - f["bytesPorSeg"]) if anterior is not None else None
        metricas = {"lambda26_ml_bytes_seg_real": round(f["bytesPorSeg"], 3),
                    "lambda26_ml_prediccion_siguiente_bps": round(f["prediccionSiguienteBps"], 3),
                    "lambda26_ml_ventana_inicio_seconds": f["ventanaSeg"]}
        if error is not None:
            metricas["lambda26_ml_error_abs_bps"] = round(error, 3)
        push(GRUPO_VIVO, metricas)
        monitor["prediccion_anterior"] = f["prediccionSiguienteBps"]
        monitor["pushes"].append((batch_id, hora(), f["ventanaTxt"], round(f["bytesPorSeg"], 1),
                                  round(f["prediccionSiguienteBps"], 1), None if error is None else round(error, 1)))

consulta = (predicciones.writeStream.foreachBatch(microlote)
            .option("checkpointLocation", CHK).trigger(processingTime="2 seconds").start())
open("./artifacts/stream_listo", "w").write(hora())     # señal para lanzar el productor
print(f"{hora()}  consulta en vivo activa; esperando ventanas de ventanas-red-py")

19:33:55  consulta en vivo activa; esperando ventanas de ventanas-red-py


### B.4 Confirmar con PromQL, mientras el stream corre, que los valores cambian

In [9]:
N_VENTANAS = 72
muestras = []
inicio = time.time()
while time.time() - inicio < 600:
    time.sleep(20)
    muestras.append((hora(), promql("lambda26_ml_bytes_seg_real"), promql("lambda26_ml_prediccion_siguiente_bps"),
                     promql("lambda26_ml_error_abs_bps"), promql("lambda26_ml_ventana_inicio_seconds")))
    if sum(1 for p in monitor["pushes"] if p[2]) >= N_VENTANAS:
        break
fin_stream = time.time()

import pandas as pd
pd.set_option("display.width", 200)
tabla = pd.DataFrame(muestras, columns=["consulta", "bytes_seg_real", "prediccion_siguiente", "error_abs", "ventana_inicio"])
tabla["ventana_campus"] = tabla["ventana_inicio"].apply(lambda s: hora(s * 1000)[:5] if s else None)
print("PromQL cada 20 s durante el stream (hora de la consulta y ventana del campus que mostraba Prometheus):")
print(tabla.drop(columns="ventana_inicio").to_string(index=False))
print(f"\nvalores distintos de lambda26_ml_bytes_seg_real en {len(tabla)} consultas: {tabla['bytes_seg_real'].nunique()}")

netlib-blas: JNI_OnLoad: dlopen(libblas.so.3) failed: libblas.so.3: cannot open shared object file: No such file or directory


PromQL cada 20 s durante el stream (hora de la consulta y ventana del campus que mostraba Prometheus):
consulta  bytes_seg_real  prediccion_siguiente  error_abs ventana_campus
19:34:15          88.077                88.252      7.296          08:10
19:34:35          53.267                67.581     41.630          08:25
19:34:55        1952.770               262.525   1859.361          08:40
19:35:15        2250.997              1188.283   1044.244          09:10
19:35:35         175.010               443.870    523.150          09:25
19:35:55         770.250               439.082    512.201          09:40
19:36:15        1538.570               578.644   1126.862          10:10
19:36:35        4071.710              1286.968   3514.124          10:25
19:36:55         116.837               196.546    306.184          10:40
19:37:15         200.127               466.667    755.401          11:10
19:37:35          78.727               112.174     71.768          11:25
19:37:55          82.

In [10]:
url = f"{PROMETHEUS}/api/v1/query_range?" + urllib.parse.urlencode({
    "query": "lambda26_ml_bytes_seg_real", "start": inicio, "end": fin_stream, "step": 5})
serie = json.load(urllib.request.urlopen(url, timeout=10))["data"]["result"][0]["values"]
print(f"query_range durante el stream: {len(serie)} puntos, {len({v for _, v in serie})} valores distintos de bytes/s real")

publicados = [p for p in monitor["pushes"] if p[2]]
print(f"\nPushes al Pushgateway: {len(publicados)} (en {len({p[0] for p in publicados})} micro-lotes)")
print(pd.DataFrame(publicados[:8] + publicados[-4:], columns=["batchId", "hora_push", "ventana", "real_Bps", "pred_t+1_Bps", "error_abs_Bps"]).to_string(index=False))

query_range durante el stream: 73 puntos, 24 valores distintos de bytes/s real

Pushes al Pushgateway: 72 (en 72 micro-lotes)
 batchId hora_push     ventana  real_Bps  pred_t+1_Bps  error_abs_Bps
       1  19:34:03 17/08 08:00      94.5          95.7            NaN
       2  19:34:06 17/08 08:05      68.2          80.8           27.5
       3  19:34:12 17/08 08:10      88.1          88.3            7.3
       4  19:34:16 17/08 08:15      64.7          83.0           23.5
       5  19:34:22 17/08 08:20      96.4          94.9           13.4
       6  19:34:26 17/08 08:25      53.3          67.6           41.6
       7  19:34:32 17/08 08:30     165.0         106.0           97.4
       8  19:34:36 17/08 08:35      82.6          93.4           23.4
      69  19:39:42 17/08 13:40     944.8         754.0          292.4
      70  19:39:48 17/08 13:45     777.8         703.8           23.8
      71  19:39:52 17/08 13:50     219.9         445.2          483.8
      72  19:39:58 17/08 13:55    

### B.5 Detener el stream y auditar lo publicado (Regla #37, después)

In [11]:
consulta.stop()
hoy = spark.read.parquet(RUTA_PRED)
audit = (hoy.withColumn("realSiguienteBps", F.lead("bytesPorSeg").over(Window.orderBy("ventana")))
            .na.drop(subset=["realSiguienteBps"])
            .withColumn("errorAbs", F.abs(F.col("prediccionSiguienteBps") - F.col("realSiguienteBps")))
            .withColumn("errorPersistencia", F.abs(F.col("bytesPorSeg") - F.col("realSiguienteBps"))))
r = audit.agg(F.count("*").alias("n"), F.avg("errorAbs").alias("mae"), F.avg("errorPersistencia").alias("maePers")).first()
errores_vivo = [p[5] for p in publicados if p[5] is not None]
print(f"Ventanas guardadas: {hoy.count()}  |  auditadas contra el real siguiente: {r['n']}")
print(f"MAE en producción (auditoría post-hoc) ... {r['mae']:.1f} B/s   (persistencia: {r['maePers']:.1f} B/s)")
print(f"Promedio del error publicado en vivo ..... {sum(errores_vivo) / len(errores_vivo):.1f} B/s  ({len(errores_vivo)} pushes)")

Ventanas guardadas: 72  |  auditadas contra el real siguiente: 71
MAE en producción (auditoría post-hoc) ... 534.5 B/s   (persistencia: 657.5 B/s)
Promedio del error publicado en vivo ..... 534.5 B/s  (71 pushes)


### B.6 El costo del atajo: el Pushgateway no sabe que el stream terminó

In [12]:
a = (hora(), promql("lambda26_ml_bytes_seg_real"), promql('time() - push_time_seconds{job="lambda26_ml_streaming"}'))
time.sleep(30)
b = (hora(), promql("lambda26_ml_bytes_seg_real"), promql('time() - push_time_seconds{job="lambda26_ml_streaming"}'))
for t, v, edad in (a, b):
    print(f"{t}  lambda26_ml_bytes_seg_real = {v}   segundos desde el último push = {edad:.0f}")
print("-> el valor no cambia, pero Prometheus lo sigue sirviendo como si fuera actual")

req = urllib.request.Request(f"{PUSHGATEWAY}/metrics/{GRUPO_VIVO}", method="DELETE")
print(f"\n{hora()}  DELETE {GRUPO_VIVO} -> HTTP {urllib.request.urlopen(req).status}")
time.sleep(20)
print(f"{hora()}  lambda26_ml_bytes_seg_real tras el DELETE = {promql('lambda26_ml_bytes_seg_real')}  (sin datos: la serie se corta)")

19:40:17  lambda26_ml_bytes_seg_real = 296.74   segundos desde el último push = 19
19:40:47  lambda26_ml_bytes_seg_real = 296.74   segundos desde el último push = 49
-> el valor no cambia, pero Prometheus lo sigue sirviendo como si fuera actual

19:40:47  DELETE job/lambda26_ml_streaming/instance/campus-trcu -> HTTP 202


19:41:07  lambda26_ml_bytes_seg_real tras el DELETE = None  (sin datos: la serie se corta)


### B.7 Regla #37 por tipo de día: ¿dónde le gana el modelo a la persistencia?

Una primera corrida en vivo cayó en sábado (15/08) y el modelo **perdió** contra la persistencia (MAE 2.1 frente a
1.5 B/s, en `evidencias-s11/corrida_sabado_15-08.ipynb`). Para saber si era mala suerte o un patrón, se compara modelo
y persistencia en **cada día** del tramo de prueba de S10, con las mismas ventanas que publica `ventanas-red-py`.

In [13]:
import numpy as np
from ventanas_campus import construir_ventanas

v = construir_ventanas("../s10-series-tiempo-red/data/TRCU.csv")
v["label"] = v.groupby("sesion").ly.shift(-1)
v = v.dropna(subset=["label"])
v = v[v.ventana >= pd.Timestamp("2026-08-12 06:55", tz="America/Lima")]        # tramo de prueba de S10
v["pred"] = modelo.transform(spark.createDataFrame(v[VARIABLES].astype(float))).select("prediction").toPandas().prediction.values
v["dia"] = v.ventana.dt.tz_convert("America/Lima").dt.strftime("%a %d/%m")
v["finDeSemana"] = v.ventana.dt.tz_convert("America/Lima").dt.dayofweek >= 5
v["errModelo"] = (np.expm1(v.pred) - np.expm1(v.label)).abs()
v["errPersistencia"] = (np.expm1(v.ly) - np.expm1(v.label)).abs()

por_dia = (v.groupby("dia", sort=False)
             .agg(ventanas=("label", "size"), bps_medio=("bytesPorSeg", "mean"),
                  mae_modelo=("errModelo", "mean"), mae_persistencia=("errPersistencia", "mean")))
por_dia["gana_modelo"] = por_dia.mae_modelo < por_dia.mae_persistencia
print(por_dia.round(1).to_string())
resumen = v.groupby("finDeSemana").agg(ventanas=("label", "size"), mae_modelo=("errModelo", "mean"), mae_persistencia=("errPersistencia", "mean"))
resumen.index = ["laborables", "fin de semana"]
resumen["mejora_%"] = 100 * (resumen.mae_persistencia - resumen.mae_modelo) / resumen.mae_persistencia
print()
print(resumen.round(1).to_string())

/usr/local/lib/python3.10/site-packages/pyspark/sql/pandas/conversion.py:687: UserWarning: createDataFrame attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)
/usr/local/lib/python3.10/site-packages/pyspark/sql/pandas/conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


           ventanas  bps_medio  mae_modelo  mae_persistencia  gana_modelo
dia                                                                      
Wed 12/08         5       23.1         6.2               3.6        False
Thu 13/08       145      317.1       227.4             275.8         True
Fri 14/08       288       40.2        10.7              13.4         True
Sat 15/08       288       22.0         1.4               1.2        False
Sun 16/08       288       22.4         1.9               1.9        False
Mon 17/08       288      290.2       212.9             283.8         True
Tue 18/08       137      129.4        98.7             107.8         True
Mon 24/08       101      522.4       432.1             589.5         True
Tue 25/08         7      637.6       580.8             621.6         True
Wed 26/08        38      994.5       885.8            1480.6         True
Thu 27/08        68      496.6       401.8             604.0         True
Mon 31/08        20     1788.3      14